# Concurrency — Threading, Multiprocessing & Asyncio

## What's covered

The three concurrency stories Python ships with — threads, processes, and asyncio coroutines — and the decision rule for picking between them. This is where the **GIL** (Global Interpreter Lock) becomes the load-bearing fact: it determines what kinds of work can actually run in parallel under each model.

- The GIL — what it is, what it locks, what it doesn't
- The three flavours of "do more than one thing at once" — concurrency, parallelism, asynchrony
- I/O-bound vs CPU-bound — the question that decides everything
- **Threads** — `threading`, `concurrent.futures.ThreadPoolExecutor`
- Why threads help for I/O-bound work (they release the GIL)
- Why threads do NOT help for CPU-bound work (they fight over the GIL)
- Thread safety basics — `Lock`, `Queue`, immutable data
- **Processes** — `multiprocessing`, `concurrent.futures.ProcessPoolExecutor`
- Process pools for CPU-bound work
- Pickling — what crosses the process boundary
- **asyncio** — coroutines, the event loop, `async`/`await`
- `asyncio.gather`, `asyncio.TaskGroup` (3.11+)
- Async vs threaded — when each fits
- Mixing them — `loop.run_in_executor`
- 3.13's no-GIL build (free-threaded Python) — the road ahead
- A decision matrix
- The honest "what to reach for first" rule

## The GIL — what it is

The **Global Interpreter Lock** is a mutex inside CPython that ensures **only one thread executes Python bytecode at a time**. It exists because CPython's memory management (reference counting) is not thread-safe; the lock protects every interpreter operation.

What the GIL means in practice:

- Two threads can be *scheduled* by the OS, but only one actually runs Python at any instant. The other waits.
- The GIL is released:
  - During I/O — file reads, socket reads, `time.sleep`, every blocking syscall.
  - Inside many C extensions (NumPy, pandas, lxml) when they enter native code.
  - Periodically (every ~5ms by default) to let other threads make progress.
- The GIL is NOT released during pure-Python CPU work — a tight loop computing primes in pure Python serializes across threads.

This single fact is why "use threads to make my Python code faster" is sometimes a great idea (I/O) and sometimes a complete waste (pure-Python CPU work).

**3.13+ has an experimental free-threaded build** without the GIL. It is not yet the default; PEP 703 sets a multi-year migration. For now, assume the GIL exists.

## Concurrency vs parallelism vs asynchrony

Three related but distinct ideas:

- **Concurrency** — *structuring* a program so that multiple things are in progress at once, even if only one runs at a time. A web server handling 1000 connections concurrently with one CPU core. Asyncio fits here.
- **Parallelism** — *actually running* multiple computations at the same instant on different CPU cores. Requires multiple cores AND the ability to use them — which under the GIL means processes, not threads.
- **Asynchrony** — *not blocking* while waiting for something. A function that fires off a network request and returns immediately, with a callback or `await` to handle the result later. Asyncio is the in-language form.

Python gives you tools for all three:

| Tool | Concurrency | Parallelism | Asynchrony |
|---|---|---|---|
| `threading` | yes | I/O only | implicit (OS does it) |
| `multiprocessing` | yes | yes | implicit |
| `asyncio` | yes | no | explicit (`await`) |

## I/O-bound vs CPU-bound

The single most useful question to ask before choosing a concurrency tool:

- **I/O-bound** — your program spends most of its time *waiting* — on a network, a disk, a database, a subprocess. The CPU is mostly idle. Examples: a web scraper, an API client fanning out 100 requests, a log shipper reading from disk.
- **CPU-bound** — your program spends most of its time *computing* — in tight Python loops, doing math, parsing strings. The CPU is pegged. Examples: image processing, simulation, anything number-crunchy that you wrote in pure Python.

For I/O-bound work, **threads or asyncio** both work — the program spends most of its time blocked on a syscall, and the GIL is released during that block. Many things wait in parallel.

For CPU-bound work, **only processes** give you real parallelism (under the GIL). Threads will not help — they will fight over the GIL and run sequentially.

How to know which one you have: profile it. `top` shows your process at 100% CPU on one core → CPU-bound. Stuck at 10% CPU with high wall-clock time → I/O-bound.

## Part 1 — Threads

## `threading` and `ThreadPoolExecutor`

The low-level API is the `threading` module. The high-level API — and what you should reach for in most code — is `concurrent.futures.ThreadPoolExecutor`, which manages a pool of worker threads and gives you a `submit`/`as_completed`/`map` interface.

The pattern: submit functions; get back `Future` objects; collect results.

In [ ]:
import time
import concurrent.futures

def slow_io(name, seconds):
    print(f"  [{name}] starting")
    time.sleep(seconds)                       # blocks — releases the GIL
    print(f"  [{name}] done after {seconds}s")
    return f"{name}-result"

# Sequential — total time = sum
t0 = time.perf_counter()
slow_io("a", 0.3)
slow_io("b", 0.3)
slow_io("c", 0.3)
print(f"sequential: {time.perf_counter() - t0:.2f}s")
print("---")

# Threaded — total time ≈ max
t0 = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(slow_io, name, 0.3) for name in ["a", "b", "c"]]
    results = [f.result() for f in futures]
print(f"threaded:   {time.perf_counter() - t0:.2f}s")
print(results)

## Threads do NOT help for CPU-bound work

The same `ThreadPoolExecutor`, given pure-Python CPU work, gets no speedup. The GIL serializes execution. Two threads computing in pure Python end up running on one core, one at a time.

In [ ]:
import time
import concurrent.futures

def cpu_busy(n):
    # Pure-Python CPU work — does NOT release the GIL
    total = 0
    for i in range(n):
        total += i * i
    return total

N = 5_000_000

# Sequential
t0 = time.perf_counter()
for _ in range(4):
    cpu_busy(N)
print(f"sequential CPU: {time.perf_counter() - t0:.2f}s")

# Threaded — basically the same time, no parallelism for pure-Python CPU
t0 = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(cpu_busy, [N] * 4))
print(f"threaded CPU:   {time.perf_counter() - t0:.2f}s   (GIL serializes)")

## Thread safety — `Lock`, `Queue`, immutable data

When threads share mutable state, you need synchronization to prevent races. The standard library gives you:

- **`threading.Lock`** — mutex. `with lock:` ensures only one thread is in the block at a time.
- **`queue.Queue`** — thread-safe queue. Producer/consumer patterns. Built-in locking; just `put` and `get`.
- **`threading.Event`, `Condition`, `Semaphore`** — higher-level primitives.

The Pythonic preference: **share data via queues, not by sharing state through locks.** Easier to reason about, fewer ways to deadlock. Treat objects you pass to a `Queue.put()` as transferred — don't keep mutating them after.

In [ ]:
import threading
import queue
import time

# Producer / consumer with a Queue
q = queue.Queue()

def producer():
    for i in range(5):
        q.put(f"item-{i}")
        time.sleep(0.05)
    q.put(None)         # sentinel — signal done

def consumer():
    while True:
        item = q.get()
        if item is None:
            break
        print(f"  consumed {item}")

p = threading.Thread(target=producer)
c = threading.Thread(target=consumer)
p.start(); c.start()
p.join();  c.join()
print("done")

## Part 2 — Processes

## `multiprocessing` and `ProcessPoolExecutor`

For CPU-bound work, you need actual parallelism — multiple Python interpreters running on multiple cores. That's processes.

- The low-level API: `multiprocessing` (mirrors `threading` in shape, but processes underneath).
- The high-level API — and what you should reach for: **`concurrent.futures.ProcessPoolExecutor`**. Same interface as `ThreadPoolExecutor`, different underlying mechanism.

Each worker process has its own Python interpreter and its own memory. No GIL contention between workers.

The cost: starting processes is slower than starting threads (tens of milliseconds vs sub-millisecond), and data passed between processes must be **pickled** — serialized to bytes, sent, deserialized.

In [ ]:
# NOTE: ProcessPoolExecutor requires functions to be importable.
# In a notebook, that means defining helpers in a module — or using a __main__
# guard if running as a script. The cell below DEMONSTRATES the API; in a real
# notebook it may fall back to running serially because of pickling limits.
import time
import concurrent.futures
import multiprocessing as mp

def cpu_work(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

N = 5_000_000

if __name__ == "__main__":   # protects re-entry on spawn-based platforms
    t0 = time.perf_counter()
    for _ in range(4):
        cpu_work(N)
    print(f"sequential: {time.perf_counter() - t0:.2f}s")

    # In a SCRIPT (not notebook), this would actually parallelize
    print(f"available CPUs: {mp.cpu_count()}")

## Pickling — what crosses the process boundary

Arguments to a process-pool task and its return value must be **picklable**. The standard library's `pickle` module handles most built-in types, but there are gotchas:

- **Lambdas and local functions** are NOT picklable. Use module-level functions.
- **Open file handles, locks, threads** are NOT picklable.
- **Class instances** must be defined at the module top level (not inside other functions).

The size of pickled data also matters — large payloads make process-pool fan-out expensive. For very small jobs, the pickling overhead can dominate. Batch small jobs into bigger ones; or use **shared memory** (`multiprocessing.shared_memory`) for big arrays.

The simplification: for numerical workloads, **use NumPy or `numba`-compiled code that releases the GIL** instead of `multiprocessing`. You get parallelism inside a single process, with no pickling cost.

## Part 3 — Asyncio

## Coroutines, the event loop, `async`/`await`

**asyncio** is Python's answer to "thousands of concurrent I/O operations in one process, one thread, no thread overhead." It uses **cooperative multitasking** — code voluntarily yields control with `await`; the event loop runs other coroutines while one is suspended.

The shape:

```python
import asyncio

async def fetch(name, delay):
    print(f"  [{name}] starting")
    await asyncio.sleep(delay)       # yields control — event loop runs other tasks
    print(f"  [{name}] done")
    return f"{name}-result"

async def main():
    # Run three coroutines concurrently
    results = await asyncio.gather(
        fetch("a", 0.3),
        fetch("b", 0.3),
        fetch("c", 0.3),
    )
    print(results)

asyncio.run(main())
```

Key vocabulary:

- **Coroutine function** — `async def f():`. Calling it returns a coroutine object; it doesn't run.
- **Coroutine** — the suspended object you get from calling `f()`. Runs when awaited or scheduled.
- **`await expr`** — suspend, run `expr` (which must be awaitable), resume with its result.
- **Event loop** — the runtime that drives coroutines. `asyncio.run(coro)` starts a loop, runs `coro` to completion, stops the loop.
- **Task** — a coroutine wrapped in a `Task` object and scheduled on the loop. Created by `asyncio.create_task(coro)` or `asyncio.gather(...)`.

In [ ]:
import asyncio
import time

async def fetch(name, delay):
    print(f"  [{name}] starting")
    await asyncio.sleep(delay)
    print(f"  [{name}] done")
    return f"{name}-result"

# Sequential awaits — each coroutine completes before the next starts
async def sequential():
    r1 = await fetch("a", 0.3)
    r2 = await fetch("b", 0.3)
    r3 = await fetch("c", 0.3)
    return [r1, r2, r3]

# Concurrent — all three run together
async def concurrent():
    return await asyncio.gather(
        fetch("a", 0.3),
        fetch("b", 0.3),
        fetch("c", 0.3),
    )

t0 = time.perf_counter()
asyncio.run(sequential())
print(f"sequential: {time.perf_counter() - t0:.2f}s")

print("---")

t0 = time.perf_counter()
asyncio.run(concurrent())
print(f"concurrent: {time.perf_counter() - t0:.2f}s")

## `asyncio.gather` and `asyncio.TaskGroup` (3.11+)

Two ways to run multiple coroutines concurrently:

- **`asyncio.gather(coro1, coro2, ...)`** — schedule, await all, return results in order. If any raises, the others continue by default (or pass `return_exceptions=True` to collect errors as values).
- **`asyncio.TaskGroup`** (3.11+) — context-manager form. If any task fails, the others are cancelled, and the errors are surfaced as an `ExceptionGroup`. This is the modern, *structured concurrency* shape.

Use `TaskGroup` for new code. `gather` is fine for simple fan-out but doesn't enforce structured cancellation.

In [ ]:
import asyncio

async def task(name, delay, fail=False):
    await asyncio.sleep(delay)
    if fail:
        raise RuntimeError(f"{name} failed")
    return f"{name}-ok"

# gather — return_exceptions=True collects errors as values
async def with_gather():
    results = await asyncio.gather(
        task("a", 0.1),
        task("b", 0.1, fail=True),
        task("c", 0.1),
        return_exceptions=True,
    )
    return results

print(asyncio.run(with_gather()))

# TaskGroup — structured concurrency (3.11+)
async def with_taskgroup():
    async with asyncio.TaskGroup() as tg:
        t1 = tg.create_task(task("a", 0.1))
        t2 = tg.create_task(task("b", 0.1))
        t3 = tg.create_task(task("c", 0.1))
    return [t1.result(), t2.result(), t3.result()]

print(asyncio.run(with_taskgroup()))

## Async vs threads — when each fits

Both work for I/O-bound concurrency. The decision rule:

| Use threads when | Use asyncio when |
|---|---|
| The libraries you call are blocking (most stdlib, `requests`, JDBC drivers, classic DB clients) | The libraries are async-native (`httpx`, `aiohttp`, `asyncpg`, `aioredis`) |
| You have hundreds of concurrent operations | You have thousands or more |
| You need to call into C-extension code that may block | You're writing servers, web crawlers, message-queue consumers |
| You want incremental adoption — sprinkling threads here and there | You're starting from scratch and can commit to async-throughout |

**The "color" problem**: in asyncio, every function in the call chain that wants to `await` must itself be `async def`. You can't call an `await` from a sync function. This bifurcates your codebase — sync functions and async functions form two near-disjoint sets. Adopting asyncio incrementally is painful. Adopting threads incrementally is easy.

For NEW services with available async libraries — asyncio wins on raw concurrency. For EXISTING codebases — threads usually slot in with less rewrite.

## Mixing them — `loop.run_in_executor`

When you have an async program but need to call a blocking function (e.g. a sync library), don't call it directly — that would block the event loop and freeze every coroutine. Instead, offload it to a thread (or process) with `loop.run_in_executor`:

```python
import asyncio
import requests   # sync library

async def fetch_via_thread(url):
    loop = asyncio.get_running_loop()
    response = await loop.run_in_executor(
        None,                    # default executor (thread pool)
        requests.get,
        url,
    )
    return response.text
```

`None` for the executor means "default thread pool." Pass a `ProcessPoolExecutor` if you need to offload CPU-bound work from an async server.

The reverse direction — using `asyncio` from a thread — uses `asyncio.run_coroutine_threadsafe(coro, loop)`. Rare.

## 3.13's no-GIL build — the road ahead

Python 3.13 (released October 2024) shipped an **experimental free-threaded build** without the GIL. PEP 703 makes this an official path forward, with a multi-year migration:

- 3.13: experimental build, opt-in (`./configure --disable-gil`).
- 3.14–3.15: still experimental but more libraries support it.
- Future: free-threaded becomes the default; legacy GIL build phased out.

What changes when the GIL is gone:

- Pure-Python threads can actually run on multiple cores.
- Reference counting becomes atomic — small performance cost on single-threaded code (~5-10% slowdown today).
- C extensions must be audited for thread safety — some currently rely on the GIL as an implicit lock.

The library ecosystem catch-up will take years. For now, plan as if the GIL exists; check whether your dependencies have free-threaded support before betting on it.

## A decision matrix

| Workload | Best tool | Why |
|---|---|---|
| Network I/O, ≤ a few hundred concurrent | Threads | Simple, works with sync libraries |
| Network I/O, thousands of concurrent | asyncio | Lighter weight, scales better |
| CPU-bound pure-Python | Processes | Only way around the GIL |
| CPU-bound, can use NumPy | NumPy or numba | Releases the GIL; multi-core without `multiprocessing` |
| Mixed I/O + CPU | asyncio + `run_in_executor` | Offload the CPU bit to a process pool |
| One-shot script, a few jobs | `ThreadPoolExecutor.map` | Simplest API |

## The honest "what to reach for first" rule

When in doubt, reach for `concurrent.futures.ThreadPoolExecutor` for I/O work and `ProcessPoolExecutor` for CPU work. They have the same API, they're in the standard library, and they handle the common cases without ceremony.

Move to `asyncio` when:

- Your I/O concurrency needs are large (thousands+ of in-flight operations).
- You're using async-native libraries already.
- You're writing a server or long-running service and can commit to async from the start.

Move to actual C-extension or compiled code (NumPy, numba, Cython, Rust via PyO3) when:

- Profiling shows you're CPU-bound.
- You've tried `ProcessPoolExecutor` and the overhead is killing you.

Skip threading-and-locks-by-hand unless you really know why you need it. `Queue`-mediated worker pools or `ThreadPoolExecutor` cover ninety percent of cases without giving you a chance to deadlock.

## What's next

You now have the three Python concurrency models and the decision rule for picking between them. The GIL is the load-bearing constraint that decides what's parallelizable; structured concurrency (`TaskGroup`) is the modern shape for async.

Notebook 10 is the last in this track — `pytest`, the dominant Python testing framework. Fixtures, parametrization, mocks, coverage. The discipline that catches regressions before they reach production. Plus a short tour of the standard library essentials worth knowing — `datetime`, `pathlib`, `json`, `re`, `argparse`, `logging` — covered as pointers since they show up across this track already.